# 🌖 LUMEN Manual Validation Checkpoint Picker

This interactive tool lets you manually select 15–30 corresponding checkpoint pairs between two lunar images (e.g. OHRC, TMC-2, IIRS, LROC) and saves the pixel coordinates to a standard JSON format.

**Purpose**: Creates held-out ground truth checkpoints for quantitative RMSE evaluation. No automatic feature matching or synthetic point generation is performed.

### Instructions:
1. Run the cells below.
2. On the interactive side-by-side display:
   - **Left Click on Left (Source)** image to mark a distinct crater/landmark.
   - **Left Click on Right (Reference)** image to mark the matching landmark.
   - Repeat across 15–30 well-distributed features over the overlap region.
3. Click **Save JSON** or run the save cell to export the coordinates.

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in python path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lumen.ingest.pds4_parser import load_binary_image
from notebooks.point_picker import load_image_any, normalize_display, ManualPointPicker

%matplotlib widget
print("Environment initialized.")

### 1. Specify Image Paths and Output Location

In [ ]:
# Set paths to source and reference images (or PDS4 XMLs)
source_path = "../data/raw/ch2_ohr_ncp_20260130T1309574436_d_img_d18/data/calibrated/20260130/ch2_ohr_ncp_20260130T1309574436_d_img_d18.xml"
reference_path = "../data/raw/ch2_tmc_ncn_20260813T0627378557_d_img_d18/data/calibrated/20260813/ch2_tmc_ncn_20260813T0627378557_d_img_d18.xml"
output_json = "../data/validation_checkpoints_ohr_tmc.json"

# Load images
src_img = load_image_any(source_path)
ref_img = load_image_any(reference_path)

print(f"Source Image loaded:    Shape={src_img.shape}, Dtype={src_img.dtype}")
print(f"Reference Image loaded: Shape={ref_img.shape}, Dtype={ref_img.dtype}")

### 2. Interactive Point Selection

- **Click Left Image (Source)** -> Then **Click Right Image (Reference)**.
- Use the bottom buttons or press `u` to Undo, `c` to Clear, `s` to Save.

In [ ]:
picker = ManualPointPicker(
    source_img=src_img,
    reference_img=ref_img,
    source_path=source_path,
    reference_path=reference_path,
    output_path=output_json,
)
picker.show()

### 3. Review and Save Recorded Checkpoints

In [ ]:
# Export recorded checkpoints to JSON
saved_path = picker.save_checkpoints()

if saved_path and Path(saved_path).exists():
    with open(saved_path) as f:
        data = json.load(f)
    print(f"Successfully saved {data['num_checkpoints']} checkpoints to: {saved_path}")
    print("\nFirst 5 pairs:")
    for p in data["pairs"][:5]:
        print(f"  #{p['id']}: Source ({p['source'][0]:.2f}, {p['source'][1]:.2f}) -> Reference ({p['reference'][0]:.2f}, {p['reference'][1]:.2f})")
else:
    print("No checkpoints saved yet. Click points on the plot above first.")

### 4. How to Use Checkpoints in LUMEN Pipeline

Pass the saved checkpoints directly into `pipeline.run(...)`:

In [ ]:
from lumen.pipeline.orchestrator import RegistrationPipeline
from lumen.config import get_settings

# Load checkpoints
with open(output_json) as f:
    chk_data = json.load(f)

# Run registration with held-out validation evaluation
# pipeline = RegistrationPipeline(config=get_settings())
# result = pipeline.run(
#     source_path=source_path,
#     reference_path=reference_path,
#     source_sensor="OHRC",
#     ref_sensor="TMC2",
#     checkpoints=(chk_data["pts_src"], chk_data["pts_ref"])
# )
# print("Validation RMSE:", result.metrics.get("rmse"))